# Fase C — validazione su silicio e chiusura mesoscopica

**FACCIATA 2.** Questo notebook *chiama e disegna*, non decide: ogni cella invoca
`phase_c.cli` o `phase_c.plots`, cioè lo stesso codice che esegue `run_phase_c.sh`.

Se una cella cominciasse a contenere logica propria, il cancello di parità
(`./run_phase_c.sh parity`) diventerebbe rosso — ed è esattamente ciò che deve fare.

Deve girare **headless** da kernel pulito:

```bash
./run_phase_c.sh notebook
```

Se non gira così, non è riproducibile — e lo si scopre adesso, non a fine campagna.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import matplotlib
matplotlib.use('Agg')          # headless: nessuna finestra, la figura resta nel notebook
import matplotlib.pyplot as plt

from phase_c import cli, plots, RESULTS
print('stadi disponibili:', cli.STADI)
print('senza scheda     :', cli.SENZA_SCHEDA)

## P1 — il plotone

Non ricalcola: **legge l'artefatto** già prodotto. Un grafico che rifacesse i conti per
conto suo mostrerebbe numeri che non stanno in nessun file, e la figura direbbe una cosa
mentre l'artefatto ne dice un'altra.

In [ ]:
P1 = os.path.join(RESULTS, 'p1.json')

if not os.path.isfile(P1):
    print('artefatto assente: eseguo P1 (circa 15 minuti)')
    cli.run_stage('p1', frontend='notebook')

for r in plots.p1_tabella(P1):
    print('N=%(N)-3d mediana %(mediana).3f  p95 %(p95).3f  max %(max).3f  |  '
          'stabili %(stabili)-7s collisioni %(collisioni)-6s TTC %(TTC_min_s).2f s' % r)

In [ ]:
ax = plots.p1_distribuzione(P1)
plt.tight_layout()
plt.show()

La riga rossa è `head-to-tail = 1`. A destra il plotone **amplifica**.

Al crescere di N la fascia attorno a 1 si svuota e la massa si polarizza: è il motivo per
cui il conteggio degli scenari stabili **non è monotono** in N, e per cui quel conteggio da
solo è una statistica che nasconde il fenomeno.

## C0–C3 — gli stadi su silicio

Sono scritti e collaudati contro il mock. Finché la scheda non è accendibile, il `cli`
**dichiara** che serve invece di restituire numeri dal mock come se fossero misure.

In [ ]:
for s in ('c0', 'c1', 'c2', 'c3'):
    try:
        cli.run_stage(s, frontend='notebook')
    except cli.SchedaAssente as e:
        print('%-3s -> %s' % (s, str(e).split('.')[0]))

## Traiettorie — accelerazione rispetto al percorso

Quando C2 avrà girato sulla scheda, `results/c2.json` conterrà le traiettorie e questa
cella le disegna: gap, velocità (ego contro leader) e accelerazione sullo stesso asse dei
tempi. La linea tratteggiata a zero sul gap è la soglia di collisione; quelle a ±16 m/s²
sull'accelerazione sono il dominio del formato di uscita `sfix13_En8`.

In [ ]:
import json
C2 = os.path.join(RESULTS, 'c2.json')
if os.path.isfile(C2):
    d = json.load(open(C2, encoding='utf-8'))['data']
    primo = sorted(d, key=int)[0]
    plots.accel_vs_traiettoria(d[primo], titolo='C2 - scenario %s' % primo)
    plt.tight_layout(); plt.show()
else:
    print('results/c2.json non c\'e ancora: C2 richiede la scheda (RUNBOOK.md).')

## Parità fra le due facciate

Lo stesso stadio, eseguito dallo script e dal notebook, deve produrre artefatti identici a
meno dei campi volatili (orario, nome della facciata, directory). La sorgente e la firma del
bitstream **non** sono volatili: un numero prodotto col mock e uno prodotto sul silicio non
sono lo stesso risultato, nemmeno quando coincidono.

```bash
./run_phase_c.sh parity
```